# Building Footprint Reader — Building Density with Persistent Caching

This notebook reads a **Shapefile** (`.shp`) of building footprints, computes
per-location **building density** within a square buffer, and caches the
result to **Google Drive** as a **GeoParquet** file so subsequent runs skip
the expensive computation entirely.

### Why GeoParquet?

The output is a `GeoDataFrame` carrying:

* the original CSV columns (Latitude, Longitude, and any other metadata),
* a Shapely `geometry` column (Point features),
* the computed `building_density_100m` column.

**GeoParquet** (`.parquet` via `gpd.GeoDataFrame.to_parquet`) is the best fit:

| Format      | Preserves geometry | CRS metadata | Read/write speed | Notes |
|-------------|--------------------|--------------|-----------------|-------|
| CSV         | ❌ (WKT string only) | ❌           | Slow            | Float re-parsing risks precision loss |
| Parquet     | ❌ (raw only)        | ❌           | Fast            | Geometry dropped or manual WKB |
| **GeoParquet** | ✅                | ✅           | **Fast**        | Purpose-built for GeoDataFrames |
| GeoPackage  | ✅                  | ✅           | Moderate        | SQLite-based; heavier for pure I/O |
| Feather     | ❌                  | ❌           | Fast            | Geometry handling awkward |

### Downstream compatibility with the GeoTIFF band-values workflow

The `extract_band_vales()` function used later for modelling produces a
DataFrame with the original CSV columns + three index columns
(`median_NDVI`, `median_NDBI`, `median_NDWI`), all aligned by row.  The cached
building-density frame here is kept **row-aligned to the same CSV in the same
order**, uses the **same `Latitude` / `Longitude` columns in EPSG:4326**, and
exposes a small helper (`load_building_density_for_concat`) that strips
duplicate columns so a safe `pd.concat([..., ...], axis=1)` just works.

In [ ]:
# cloning repo
!git clone https://github.com/chase-kusterer/bc-II.git
%cd bc-II
!cat .env_bin.txt
!pip install -r /content/bc-II/requirements.txt

In [ ]:
# Suppress non-critical warnings
import warnings
warnings.filterwarnings('ignore')

# File management
import os
import zipfile

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Geospatial vector data handling (e.g., shapefiles, GeoParquet)
import geopandas as gpd
from shapely.geometry import box, Point

# Progress bar for loops
from tqdm import tqdm

In [ ]:
# ---------------------------------------------------------------------------
# Mount Google Drive so it appears as a local filesystem path in Colab.
# You will be prompted to authorise access the first time.
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# Define the Drive output directory for cached building-density frames.
# Mirrors the GeoTIFF folder structure so related assets live together.
# TODO: Adjust the folder path to match your Drive structure.
# ---------------------------------------------------------------------------
output_dir = "/content/drive/MyDrive/Business Challenge II/BuildingDensity"
os.makedirs(output_dir, exist_ok=True)

## Input Data — Building Footprints Shapefiles

Three Shapefiles — *Chile.shp*, *Brazil.shp*, and *Sierra Leone.shp* — cover
Santiago, Rio de Janeiro, and Sierra Leone respectively, sourced from the
[3D-GloBFP: The First Global Three-Dimensional Building Footprint Dataset](https://essd.copernicus.org/articles/16/5357/2024/essd-16-5357-2024-discussion.html).
They provide spatial polygons used to compute structural features around each
UHI observation point.

In [ ]:
# ---------------------------------------------------------------------------
# OPTIONAL: unzip the raw building-footprint archive if it is still compressed.
# After the first run, the extracted .shp files live in `target_dir` and this
# cell can stay commented out.
# ---------------------------------------------------------------------------
target_dir = "/content/drive/My Drive/Business Challenge II/Building/"   # TODO: set to the folder where .shp files should live
# zip_path  = "/content/drive/MyDrive/Business Challenge II/Notebook 3/Building_Footprint_Data.zip"
# with zipfile.ZipFile(file=zip_path, mode='r') as zip_ref:
#     zip_ref.extractall(path=target_dir)

## Building Density Computation with Persistent Cache

The core function below performs three tasks in order:

1. **Cache check** — if a GeoParquet file already exists at the target Drive
   path and `force_recompute=False`, load and return it immediately.  This is
   the mechanism that avoids re-running the slow spatial join in future
   sessions.
2. **Compute** — project points and buildings to EPSG:3857 (metres), buffer
   each point by `buffer_m`, count intersecting buildings, divide by area.
3. **Save** — reproject geometry back to EPSG:4326 so the saved file is
   internally consistent with the WGS84 `Latitude`/`Longitude` columns used
   by the GeoTIFF band-value workflow, then write to GeoParquet.

In [ ]:
# ===========================================================================
# Building density with GeoParquet caching
# ===========================================================================
def compute_building_density(
    csv_path,
    buildings_shp_path,
    cache_path,
    lat_col="Latitude",
    lon_col="Longitude",
    buffer_m=100,
    force_recompute=False,
):
    """
    Compute building density (buildings per m²) within a square buffer around
    each point in a CSV, caching the result as a GeoParquet file on Drive.

    Parameters
    ----------
    csv_path : str
        Path to the input CSV of observation points (must contain lat_col and
        lon_col in WGS84 / EPSG:4326).
    buildings_shp_path : str
        Path to the shapefile of building footprint polygons.
    cache_path : str
        Target .parquet path on Google Drive where the result is saved.  If
        this file already exists and `force_recompute=False`, the cached
        GeoDataFrame is loaded and returned immediately.
    lat_col, lon_col : str
        Column names holding latitude and longitude in `csv_path`.
    buffer_m : float
        Side length of the square buffer around each point, in metres.
    force_recompute : bool
        If True, ignore any existing cache and re-compute from scratch.

    Returns
    -------
    geopandas.GeoDataFrame
        Original CSV columns + Shapely Point `geometry` (EPSG:4326) +
        `building_density_100m` column (buildings per square metre).

    Notes
    -----
    Row order is preserved from the input CSV so the result is safe to
    `pd.concat(..., axis=1)` with other frames derived from the same CSV in
    the same order (e.g. the GeoTIFF-band-value frame).
    """

    # -----------------------------------------------------------------------
    # Step 1 — Cache hit?  Load and return without recomputing.
    # -----------------------------------------------------------------------
    if os.path.exists(cache_path) and not force_recompute:
        print(f"[cache hit] Loading existing GeoParquet: {cache_path}")
        return gpd.read_parquet(cache_path)

    print(f"[cache miss] Computing building density from scratch...")

    # -----------------------------------------------------------------------
    # Step 2 — Load CSV and promote to GeoDataFrame in WGS84 (EPSG:4326).
    # -----------------------------------------------------------------------
    df = pd.read_csv(csv_path)
    df['geometry'] = df.apply(
        lambda row: Point(row[lon_col], row[lat_col]), axis=1
    )
    gdf_points = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

    # -----------------------------------------------------------------------
    # Step 3 — Load building footprints and project to metric CRS for the
    # buffer/area calculations.  Web Mercator (EPSG:3857) uses metres, which
    # is required for meaningful distance and area math.
    # -----------------------------------------------------------------------
    gdf_buildings = gpd.read_file(buildings_shp_path)
    if gdf_buildings.crs != "EPSG:3857":
        gdf_buildings = gdf_buildings.to_crs(epsg=3857)

    # Reproject points to the same metric CRS so buffering matches the
    # buildings' coordinate system.
    gdf_points_3857 = gdf_points.to_crs(epsg=3857)

    # -----------------------------------------------------------------------
    # Step 4 — Loop over each point, count buildings intersecting a square
    # buffer of side `buffer_m` centred on the point, and compute density.
    # -----------------------------------------------------------------------
    building_densities = []
    for _, row in tqdm(
        gdf_points_3857.iterrows(),
        total=len(gdf_points_3857),
        desc="Calculating building density",
    ):
        point_geom = row.geometry

        # Square buffer (axis-aligned bounding box) centred on the point.
        buffer_geom = box(
            point_geom.x - buffer_m / 2, point_geom.y - buffer_m / 2,
            point_geom.x + buffer_m / 2, point_geom.y + buffer_m / 2,
        )

        # Select buildings whose geometries intersect the buffer.
        buildings_in_buffer = (
            gdf_buildings[gdf_buildings.geometry.intersects(buffer_geom)].copy()
        )
        # Zero-width buffer is a common fix for invalid polygon geometries.
        buildings_in_buffer["geometry"] = buildings_in_buffer.geometry.buffer(0)

        area_m2   = buffer_geom.area            # square metres (EPSG:3857)
        bldg_count = len(buildings_in_buffer)
        density   = bldg_count / area_m2 if area_m2 > 0 else 0

        building_densities.append(density)

    # Attach the computed densities to the ORIGINAL (WGS84) GeoDataFrame so
    # the saved geometry column is in EPSG:4326 — consistent with the
    # Latitude/Longitude columns and with the GeoTIFF band-values workflow.
    gdf_points["building_density_100m"] = building_densities

    # -----------------------------------------------------------------------
    # Step 5 — Persist to Google Drive as GeoParquet.
    # os.makedirs handles nested folders if the user points at a sub-path.
    # -----------------------------------------------------------------------
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    gdf_points.to_parquet(cache_path)
    print(f"[saved] GeoParquet written to: {cache_path}")
    print(f"        rows={len(gdf_points)}, CRS={gdf_points.crs}")

    return gdf_points

## Example Usage — Santiago (Chile)

The first call computes and caches; every subsequent call in a new Colab
runtime loads from the cached file in milliseconds.

**TODO:** Update `csv_path`, `buildings_shp_path`, and the cache filename
below to match your inputs.

In [ ]:
# ---------------------------------------------------------------------------
# Chile / Santiago building density
# TODO: fill in the two input paths and confirm the cache filename.
# ---------------------------------------------------------------------------
chile_cache_path = os.path.join(output_dir, "Santiago_building_density_100m.parquet")

chile_buildings_data = compute_building_density(
    csv_path="/content/bc-II/Data/sample_chile_uhi_data.csv",              # TODO: path to Santiago UHI CSV
    buildings_shp_path=target_dir+"Chile Building Footprints/Chile_clipped.shp",    # TODO: path to Chile.shp
    cache_path=chile_cache_path,
    buffer_m=100,
    force_recompute=False,                # flip to True to rebuild the cache
)

# Inspect the first few rows — should include Lat/Lon, geometry, and
# building_density_100m.
chile_buildings_data.head()